# เทรน wake word "hey luna" (openWakeWord) บน Colab

Colab ตอนนี้เป็น Python 3.13 แต่ `piper-phonemize` (ใช้สร้างเสียงสังเคราะห์) รองรับถึง 3.11
notebook นี้จึงสร้าง **Python 3.10 แยกไว้ที่ `/content/py310`** แล้วรันทุกขั้นผ่านตัวนั้น พร้อม pin เวอร์ชันให้ตรงกับที่ openWakeWord ออกแบบไว้

**ก่อนเริ่ม:** Runtime → Change runtime type → **T4 GPU** แล้วรันทีละ cell จากบนลงล่าง (หรือ Runtime → Run all)

## 1) ตั้งค่า (แก้ได้)

In [ ]:
TARGET_PHRASE = "hey luna"
MODEL_NAME = "hey_luna"
N_SAMPLES = 3000        # จำนวนเสียงสังเคราะห์ต่อชุด ยิ่งมากยิ่งแม่นแต่นานขึ้น (ทดลองเร็วๆ ใช้ 1000)
N_SAMPLES_VAL = 1000
STEPS = 20000
# คำที่คล้ายแต่ไม่ใช่ wake word ช่วยลด false trigger
CUSTOM_NEGATIVES = ["hey lena", "hey lunar", "hey laura", "hey you", "tuna", "hey luke"]

## 2) ติดตั้ง Python 3.10 + dependencies (~5-10 นาที)

In [ ]:
%cd /content
import os
PY = "/content/py310/bin/python"
os.environ["MPLBACKEND"] = "Agg"  # backend ของ Colab ไม่มีใน Python 3.10 แยก

!pip install -q uv
if not os.path.exists(PY):
    !uv venv -q --python 3.10 /content/py310

if not os.path.exists("piper-sample-generator"):
    !git clone -q https://github.com/rhasspy/piper-sample-generator
# pin เวอร์ชันก่อนย้ายไปเป็น package (มี generate_samples.py ที่ train.py ใช้)
!cd piper-sample-generator && git checkout -q 9c1019c932eda098aa70d63cdeac803dea983f53
if not os.path.exists("piper-sample-generator/models/en_US-libritts_r-medium.pt"):
    !wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt

if not os.path.exists("openwakeword"):
    !git clone -q https://github.com/dscripka/openwakeword

!uv pip install -q --python {PY} \
    "torch==2.1.2" "torchaudio==2.1.2" "numpy<2" "scipy==1.11.4" \
    piper-phonemize==1.1.0 webrtcvad mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 \
    speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6 \
    pronouncing==0.2.0 deep-phonemizer==0.0.19 \
    datasets==2.14.6 "pyarrow<15" "huggingface_hub<0.25" "fsspec[http]<=2023.10.0" \
    soundfile "librosa<0.11" onnx==1.14.0 "onnxruntime<1.20" pyyaml tqdm scikit-learn requests \
    "setuptools<81"  # webrtcvad / pronouncing ยังใช้ pkg_resources
!uv pip install -q --python {PY} --no-deps -e ./openwakeword

# โมเดล feature extractor ของ openWakeWord
res = "openwakeword/openwakeword/resources/models"
os.makedirs(res, exist_ok=True)
for f in ["embedding_model.onnx", "melspectrogram.onnx"]:
    if not os.path.exists(f"{res}/{f}"):
        !wget -q -O {res}/{f} https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/{f}

# patch: train.py แปลงเป็น tflite ทุกครั้งเพราะ default="False" เป็น string (เราใช้ onnx อย่างเดียว)
!sed -i 's|if args.convert_to_tflite:|if args.convert_to_tflite is True:|' openwakeword/openwakeword/train.py

!{PY} -c "import torch, torchaudio; from piper_phonemize import phonemize_espeak; import webrtcvad, pronouncing, openwakeword.data, speechbrain, audiomentations, torch_audiomentations, acoustics, datasets; print('OK  python', __import__('sys').version.split()[0], ' torch', torch.__version__, ' cuda', torch.cuda.is_available())"

## 3) ทดสอบฟังเสียงสังเคราะห์ (ไม่บังคับ)
ถ้าเสียงไม่เหมือน "ลูน่า" ลองเปลี่ยน `TARGET_PHRASE` เป็น `"hey loona"` แล้วรัน cell 1 และ cell นี้ใหม่

In [ ]:
!rm -rf /content/preview && {PY} piper-sample-generator/generate_samples.py "{TARGET_PHRASE}" --max-samples 4 --output-dir /content/preview
from IPython.display import Audio, display
import glob
for f in sorted(glob.glob("/content/preview/*.wav")):
    display(Audio(f))

## 4) ดาวน์โหลดข้อมูล (~20-30 นาที, ไฟล์ features ~17 GB)

In [ ]:
%%writefile /content/prepare_data.py
import os
import subprocess
from pathlib import Path

import datasets
import numpy as np
import scipy.io.wavfile
from tqdm import tqdm

def save(path, arr):
    scipy.io.wavfile.write(path, 16000, (np.asarray(arr) * 32767).astype(np.int16))

# Room impulse responses (เสียงก้องห้อง)
os.makedirs("mit_rirs", exist_ok=True)
if len(os.listdir("mit_rirs")) < 200:
    for row in tqdm(datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True), desc="RIR"):
        save(os.path.join("mit_rirs", row["audio"]["path"].split("/")[-1]), row["audio"]["array"])

# AudioSet (เสียงรบกวนทั่วไป) -> 16 kHz mono ด้วย ffmpeg
os.makedirs("audioset_16k", exist_ok=True)
if len(os.listdir("audioset_16k")) < 100:
    flacs = [str(p) for p in Path("audioset").glob("**/*.flac")]
    print("AudioSet flac:", len(flacs))
    for f in tqdm(flacs, desc="AudioSet"):
        out = os.path.join("audioset_16k", Path(f).stem + ".wav")
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", f, "-ac", "1", "-ar", "16000", out])

# Free Music Archive (เพลง) 1 ชั่วโมง
os.makedirs("fma", exist_ok=True)
if len(os.listdir("fma")) < 100:
    fma = iter(datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
               .cast_column("audio", datasets.Audio(sampling_rate=16000)))
    for _ in tqdm(range(3600 // 30), desc="FMA"):
        row = next(fma)
        save(os.path.join("fma", row["audio"]["path"].split("/")[-1].replace(".mp3", ".wav")), row["audio"]["array"])
print("data ready")

In [ ]:
import os
os.makedirs("audioset", exist_ok=True)
if not os.path.exists("audioset/audio"):
    !wget -q -O audioset/bal_train09.tar https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
    !cd audioset && tar -xf bal_train09.tar && rm bal_train09.tar

!{PY} /content/prepare_data.py

HF = "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main"
!wget -q -c {HF}/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -q -c {HF}/validation_set_features.npy
!ls -lh *.npy

## 5) สร้าง config

In [ ]:
import yaml
config = yaml.safe_load(open("openwakeword/examples/custom_model.yml"))
config.update({
    "target_phrase": [TARGET_PHRASE],
    "model_name": MODEL_NAME,
    "custom_negative_phrases": CUSTOM_NEGATIVES,
    "n_samples": N_SAMPLES,
    "n_samples_val": N_SAMPLES_VAL,
    "steps": STEPS,
    "target_accuracy": 0.6,
    "target_recall": 0.25,
    "rir_paths": ["./mit_rirs"],
    "background_paths": ["./audioset_16k", "./fma"],
    "background_paths_duplication_rate": [1, 1],
    "false_positive_validation_data_path": "validation_set_features.npy",
    "feature_data_files": {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"},
})
yaml.dump(config, open("my_model.yaml", "w"))
print(yaml.dump(config, allow_unicode=True))

## 6) เทรน
แต่ละขั้นรันซ้ำได้ ถ้าขั้นสร้างเสียง error กลางทาง รันซ้ำจะทำต่อจากเดิม

In [ ]:
# 6.1 สร้างเสียงสังเคราะห์ (~20-40 นาทีบน T4 ที่ N_SAMPLES=3000)
%cd /content
PY = "/content/py310/bin/python"  # กำหนดซ้ำเผื่อ session ถูกรีสตาร์ท
import os
os.environ["MPLBACKEND"] = "Agg"  # backend ของ Colab ไม่มีใน Python 3.10 แยก
!{PY} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

In [ ]:
# 6.2 เติมเสียงรบกวน/เสียงก้อง แล้วแปลงเป็น features
!{PY} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

In [ ]:
# 6.3 เทรนโมเดล
!{PY} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

## 7) ดาวน์โหลดโมเดล
นำไฟล์ไปไว้ที่ `SpeechtoText/models/wakeword/` แล้วตั้ง `wake.model: models/wakeword/hey_luna.onnx` ใน `config.yaml`

In [ ]:
# ไม่พึ่งตัวแปรจาก cell 1 เผื่อ session ถูกรีสตาร์ท
import glob
from google.colab import files
models = glob.glob("/content/my_custom_model/*.onnx")
print(models or "ไม่พบไฟล์ .onnx — ขั้น 6.3 ยังไม่เสร็จ หรือ runtime ถูกรีเซ็ตจนไฟล์หาย")
for m in models:
    files.download(m)